In [1]:
import os
import sys
import dill
import pprint
from pathlib import Path
import itertools
import pathos
sys.path.append('/Users/leonardo.tessarolo/git/bayesian_bss/')

import math
import numpy as np
import jax.numpy as jnp
import functools
import pandas as pd
from numpyro.diagnostics import gelman_rubin
import scipy.stats as st
import time
import gc

from matplotlib import pyplot as plt
# import seaborn as sns

from dataclasses import dataclass, field
from typing import Callable, Dict, List, Optional, Sequence, Union
import warnings

import jax
jax.config.update("jax_enable_x64", True)
import numpy as np
import pandas as pd

try:
    from tqdm import tqdm
    _HAS_TQDM = True
except ImportError:
    _HAS_TQDM = False

from src import MultivariateTPrior, LocalSensitivityScores, LoggingUtilities, StudentTSource, ContinuousExperimentExecutor
np.random.seed(100)

In [2]:
# Whether or not to create config
EXECUTE=True

# Whether or not to create folder structure
CREATE_FOLDER_STRUCTURE=True

# Experiment name
EXPERIMENT_NAME='importance_sampling_student_t'

# Folder which will contain output directory tree
OUTPUT_DIR='./output'
base_output_path=Path(OUTPUT_DIR)

# Create experiment directory
experiment_dir = base_output_path / EXPERIMENT_NAME
LoggingUtilities.configure_logging(
    log_dir=experiment_dir,
    log_type='continuous'
)

# 1. Configurations

In [3]:
# Number of sources and observations
NSOURCES=2
NOBS=1000

# Mixing matrix configuration
A = np.array([
    [math.sqrt(2)/2, -math.sqrt(2)/2],
    [math.sqrt(2)/2, math.sqrt(2)/2]
])

# Ideal separating matrix
B = np.linalg.inv(A)

if EXECUTE:
    # Execution configurations
    exec_configs = {
        'general': {
            # General output directory
            'base_output_path': base_output_path,
            # Experiment name for folder structure
            'experiment_name': EXPERIMENT_NAME,
            # Experiment directory
            'experiment_dir': experiment_dir,
            # Whether or not to create folder structure
            'create_folder_structure': CREATE_FOLDER_STRUCTURE,
            # Whether or not to save graphs
            'save_graphs': False,
            # Number of sources
            'n_sources': NSOURCES,
            # Number of observations in each realization
            'n_obs': NOBS,
            # Number of parallel workers
            'n_workers': os.cpu_count(),
            # Number of parallel realizations to run
            'n_realizations': 30,
            # Mixing matrix
            'A': A,
            # Separating matrix
            'B': B,
            # Initial condition
            'initial_B': B,
            # Whether or not to use a normalized posterior
            'normalize_posterior': True,
            # Whether or not to have a progress bar for each initialization test case
            'initialization_progress_bar': False,
            # Whether or not to have a progress bar for each realization
            'realizations_progress_bar': True
        },
        'prior_variation':{
            'mode':{
                'base_prior': MultivariateTPrior(
                    center=0.5*(B + np.eye(NSOURCES)),
                    scale=np.sqrt(1/3),
                    df=3,
                    use_jax=True
                ),
                'target_param_values': np.linspace(
                    start=0,
                    stop=1,
                    num=31
                ),
                'target_param_fn': lambda target_params: [
                    MultivariateTPrior(
                        center=(1-p)*B + p*np.eye(NSOURCES),
                        scale=np.sqrt(1/3),
                        df=3,
                        use_jax=True
                    ) for p in target_params
                ],
                'score_fn': LocalSensitivityScores.prior_mode_score(
                    direction=np.eye(NSOURCES)-B
                )
            },
            'scale':{
                'base_prior': MultivariateTPrior(
                    center=B,
                    scale=5E-1,
                    df=3,
                    use_jax=True
                ),
                'target_param_values': np.logspace(
                    start=-1,
                    stop=1,
                    num=31,
                    base=10
                ),
                'target_param_fn': lambda target_params: [
                    MultivariateTPrior(
                        center=B,
                        scale=p,
                        df=3,
                        use_jax=True
                    ) for p in target_params
                ],
                'score_fn': LocalSensitivityScores.prior_scale_score
            },
            'tail_heaviness': {
                'base_prior': MultivariateTPrior(
                    center=B,
                    scale=np.sqrt(1/3),
                    df=3,
                    use_jax=True
                ),
                'target_param_values': range(3, 13),
                'target_param_fn': lambda target_params: [
                    MultivariateTPrior(
                        center=B,
                        scale=np.sqrt((p - 2) / p),
                        df=p,
                        use_jax=True
                    )
                    for p in target_params
                ],

                'score_fn': LocalSensitivityScores.prior_df_score
            },
        },
        'source_model_variation': {
            'mode':{
                'param_name': 'mu',
                'base_prior': MultivariateTPrior(
                    center=B,
                    scale=np.sqrt(1/3),
                    df=3,
                    use_jax=True
                ),
                'target_param_values': np.linspace(
                    start=-0.15,
                    stop=0.15,
                    num=31
                ),
                'target_param_fn': lambda baseline_objs, target_params, param_name: [
                    setattr(
                        b_obj,
                        param_name,
                        p
                    ) for b_obj, p in zip(
                        baseline_objs,
                        target_params
                    )
                ],
                'score_fn': LocalSensitivityScores.source_mu_score
            },
            'scale':{
                'param_name': 'sigma',
                'base_prior': MultivariateTPrior(
                    center=B,
                    scale=np.sqrt(1/3),
                    df=3,
                    use_jax=True
                ),
                'target_param_values': np.linspace(
                    0.97,
                    1.03,
                    31
                ),
                'target_param_fn': lambda baseline_objs, target_params, param_name: [
                    setattr(
                        b_obj,
                        param_name,
                        b_obj.sigma*p
                    ) for b_obj, p in zip(
                        baseline_objs,
                        target_params
                    )
                ],
                'score_fn': LocalSensitivityScores.source_sigma_score
            },
            'tail_heaviness': {
                'param_name': 'df',
                'base_prior': MultivariateTPrior(
                    center=B,
                    scale=np.sqrt(1/3),
                    df=3,
                    use_jax=True
                ),
                'target_param_values': np.linspace(
                    0.95,
                    1.05,
                    31
                ),
                'target_param_fn': lambda baseline_objs, target_params, param_name: [
                    (
                        setattr(
                            b_obj,
                            'df',
                            float(b_obj.df) * r
                        ),                       # df primeiro
                        setattr(
                            b_obj,
                            'sigma',
                            np.sqrt((b_obj.df - 2) / b_obj.df)
                        )      # usa o df novo
                    ) for b_obj, r in zip(baseline_objs, target_params)
                ],
                'score_fn': LocalSensitivityScores.source_df_score
            },
        },
        'prior_source_model_variation':{},
        'mcmc': {
            # Initial condition noise
            'starting_distribution': lambda: st.t.rvs(
                loc=0,
                scale=0.05,
                df=3,
                size=B.shape
            ),
            # Exploration variance for MCMC
            'exploration_var': 1E-4,
            # Number of samples to generate
            'n_samples': 30000,
            # Number of parallel chains for MCMC
            'parallel_chains': 4,
            # Maximum iterations for MCMC
            'max_it': 20000,
            # Convergence threshold for R-hat diagnostic
            'R_hat_thresh': 1.01,
            # Interval at which R-hat will be evaluated
            'R_hat_evaluation_step': 500,
            # Number of consecutive convergent evaluations necessary
            'R_hat_persistance': 2,
            # Samples which will be discarded
            'R_hat_minimum_burn_in': 5000,
            # Whether or not to adjust exploration variance inside MCMC
            'auto_adjust_exploration_var': True,
            # Whether or not to show progress bars
            'progress_bar': False,
            # Target acceptance probability, in case auto adjust is enabled
            'target_accept_prob': 0.5,

        },
        'sources':{
            'rs1': StudentTSource(
                mu=0,
                sigma=math.sqrt(1/3),
                df=3,
                use_jax=True
            ),
            'rs2': StudentTSource(
                mu=0,
                sigma=math.sqrt(1/2),
                df=4,
                use_jax=True
            ),
            'rs3': StudentTSource(
                mu=0,
                sigma=math.sqrt(3/5),
                df=5,
                use_jax=True
            ),
            'rs4': StudentTSource(
                mu=0,
                sigma=math.sqrt(2/3),
                df=6,
                use_jax=True
            ),
            'rs5': StudentTSource(
                mu=0,
                sigma=math.sqrt(5/7),
                df=7,
                use_jax=True
            ),
            'rs6': StudentTSource(
                mu=0,
                sigma=math.sqrt(3/4),
                df=8,
                use_jax=True
            ),
            'rs7': StudentTSource(
                mu=0,
                sigma=math.sqrt(7/9),
                df=9,
                use_jax=True
            ),
            'rs8': StudentTSource(
                mu=0,
                sigma=math.sqrt(4/5),
                df=10,
                use_jax=True
            ),
            'rs9': StudentTSource(
                mu=0,
                sigma=math.sqrt(9/11),
                df=11,
                use_jax=True
            ),
            'rs10': StudentTSource(
                mu=0,
                sigma=math.sqrt(5/6),
                df=12,
                use_jax=True
            )
        }
    }
else:
    with (experiment_dir/'execution_config.pkl').open('rb') as f:
        exec_configs = dill.load(f)

pprint.pp(exec_configs)

{'general': {'base_output_path': PosixPath('output'),
             'experiment_name': 'importance_sampling_student_t',
             'experiment_dir': PosixPath('output/importance_sampling_student_t'),
             'create_folder_structure': True,
             'save_graphs': False,
             'n_sources': 2,
             'n_obs': 1000,
             'n_workers': 8,
             'n_realizations': 30,
             'A': array([[ 0.70710678, -0.70710678],
       [ 0.70710678,  0.70710678]]),
             'B': array([[ 0.70710678,  0.70710678],
       [-0.70710678,  0.70710678]]),
             'initial_B': array([[ 0.70710678,  0.70710678],
       [-0.70710678,  0.70710678]]),
             'normalize_posterior': True,
             'initialization_progress_bar': False,
             'realizations_progress_bar': True},
 'prior_variation': {'mode': {'base_prior': <src.prior.MultivariateTPrior object at 0x11c8c8050>,
                              'target_param_values': array([0.        , 0.03333

# 2. Folder Structure

In [4]:
# Initialize folder structure, if so specified
if CREATE_FOLDER_STRUCTURE:
    if exec_configs['general']['create_folder_structure']:
        # Creates base output path and experiment dir
        if not base_output_path.is_dir():
            base_output_path.mkdir()
        if not experiment_dir.is_dir():
            experiment_dir.mkdir()

        # Creates folders for individual realizations
        for r in range(exec_configs['general']['n_realizations']):
            # Overall folder for realization
            realization_dir = experiment_dir / str(r)
            if not realization_dir.is_dir():
                realization_dir.mkdir()
            # Initialize success flag
            with (realization_dir/'success_flag.pkl').open('wb') as f:
                    dill.dump('UNFINISHED', f)
        

if EXECUTE:
     with (experiment_dir/'execution_config.pkl').open('wb') as f:
        dill.dump(exec_configs, f)


# 3. Executar Experimento

In [5]:
if EXECUTE:
    ex = ContinuousExperimentExecutor(cfg=exec_configs)
    ex.run()

####################################################################################################
Total realizations: 30
Finished realizations: 0
Unfinished realizations: 30
####################################################################################################
Executando experimento


100%|██████████| 30/30 [20:02:26<00:00, 2404.87s/it]  
